# 실습 3: 얼굴 탐지 (Face Detection)

## 학습 목표
- Amazon Rekognition의 `detect_faces` API로 얼굴을 탐지하고 속성을 분석합니다.
- 감정(Emotion), 나이 추정, 성별, 표정 등의 FaceDetail 속성을 파싱합니다.

## API 개요
`detect_faces`는 이미지에서 얼굴을 탐지하고 다양한 속성을 반환합니다.
- **Attributes='ALL'**: 감정, 나이, 성별, 표정 등 전체 속성 반환
- **Attributes='DEFAULT'**: BoundingBox, Confidence 등 기본 속성만 반환
- **반환값**: `response['FaceDetails']` — 각 얼굴의 상세 정보 리스트

### FaceDetail 주요 키
| 키 | 설명 | 형식 |
|---|---|---|
| `AgeRange` | 추정 나이 범위 | `{'Low': int, 'High': int}` |
| `Gender` | 성별 | `{'Value': str, 'Confidence': float}` |
| `Emotions` | 감정 목록 | `[{'Type': str, 'Confidence': float}]` |
| `Smile` | 웃음 여부 | `{'Value': bool, 'Confidence': float}` |
| `Eyeglasses` | 안경 착용 | `{'Value': bool, 'Confidence': float}` |


In [ ]:
import boto3
import os
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# ✅ [제공 코드]
rekognition = boto3.client('rekognition', region_name='ap-northeast-2')
IMAGE_DIR = './images/'
image_filename = 'lab03.jpg'
image_path = os.path.join(IMAGE_DIR, image_filename)

def load_image_bytes(path):
    with open(path, 'rb') as f:
        return f.read()

image_bytes = load_image_bytes(image_path)
print(f"이미지 경로: {image_path}")

## ✏️ TODO 1: detect_faces API 호출

Attributes='ALL'을 사용하여 전체 얼굴 속성을 분석하세요.


In [ ]:
# ✏️ TODO 1: detect_faces API를 호출하세요
response = rekognition._____( # ← detect_faces
    Image={'Bytes': _____},  # ← image_bytes
    Attributes=[_____]       # ← 'ALL'
)

print(f"탐지된 얼굴 수: {len(response['FaceDetails'])}개")

## ✏️ TODO 2: 얼굴 속성 파싱 및 출력

각 얼굴의 나이, 성별, 감정(상위 3개)을 출력하는 코드를 완성하세요.


In [ ]:
# ✏️ TODO 2: 얼굴 속성 정보를 출력하세요
for i, face in enumerate(response['FaceDetails']):
    print(f"\n얼굴 #{i+1}")
    print("-" * 40)
    
    # 나이 범위 출력
    age = face[_____]  # ← 'AgeRange' 키
    print(f"  추정 나이: {age[_____]}~{age[_____]}세")  # ← 'Low', 'High' 
    
    # 성별 출력
    gender = face[_____]  # ← 'Gender'
    print(f"  성별: {gender['Value']} (신뢰도: {gender['Confidence']:.1f}%)")
    
    # 감정 분석 - 상위 3개 정렬 출력
    emotions = sorted(
        face[_____],           # ← 'Emotions' 
        key=lambda x: x[_____], # ← 'Confidence'
        reverse=_____           # ← True
    )
    print("  감정 분석 (상위 3개):")
    for emotion in emotions[:3]:
        print(f"    - {emotion['Type']:<12} {emotion['Confidence']:.1f}%")
    
    # 표정 속성
    print(f"  웃음: {face['Smile']['Value']} ({face['Smile']['Confidence']:.1f}%)")
    print(f"  안경: {face['Eyeglasses']['Value']} ({face['Eyeglasses']['Confidence']:.1f}%)")

## ✏️ TODO 3: BoundingBox로 얼굴 영역 표시

탐지된 얼굴 주위에 녹색 사각형을 그리세요.


In [ ]:
# ✏️ TODO 3: 얼굴 BoundingBox를 이미지에 표시하세요
img = Image.open(image_path)
draw = ImageDraw.Draw(img)
w, h = img.size

for i, face in enumerate(response['FaceDetails']):
    box = face[_____]  # ← 'BoundingBox'
    
    left   = box[_____] * w  # ← 'Left'
    top    = box[_____] * h  # ← 'Top'
    right  = left + box[_____] * w  # ← 'Width'
    bottom = top  + box[_____] * h  # ← 'Height'
    
    draw.rectangle([left, top, right, bottom], outline='lime', width=3)
    top_emotion = sorted(face['Emotions'], key=lambda x: x['Confidence'], reverse=True)[0]
    draw.text((left, top-20), f"Face #{i+1}: {top_emotion['Type']}", fill='lime')

plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Face Detection 결과')
plt.show()

## 💡 심화 도전
1. Attributes를 'DEFAULT'로 바꾸면 어떤 정보가 사라지나요?
2. 탐지된 모든 얼굴 중 가장 행복한(HAPPY) 얼굴을 찾아 출력해보세요.
3. 여러 명이 있는 이미지로 테스트하고 각 얼굴에 감정을 레이블로 표시해보세요.
